In [34]:
import pandas as pd
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

In [ ]:
#train and predict with best model
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

log_reg = LogisticRegression(max_iter= 5000, random_state= 42, class_weight= 'balanced')

log_reg.fit(X_train, y_train)

predictions = log_reg.predict(X_test)
probabilities = log_reg.predict_proba(X_test)[:, 1]

print(classification_report(y_test, predictions))
print(f'ROC AUC Score: {roc_auc_score(y_test, probabilities)}')

              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1035
           1       0.50      0.78      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.73      0.75      1409

ROC AUC Score: 0.8362293006794287


In [ ]:
#use GridSearch to find best hyperparameters for F1-score
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10,],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': [None, 'balanced']
    }

log_reg_grid = LogisticRegression(max_iter= 5000, random_state= 42)

gridsearch = GridSearchCV(log_reg_grid, param_grid, scoring= 'f1', cv= 5, n_jobs= -1, verbose= 2)

gridsearch.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l2'


In [37]:
gridsearch.best_params_

{'C': 0.1, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}

In [38]:
gridsearch.best_score_

np.float64(0.6310218136880715)

In [39]:
gridsearch.best_estimator_

,penalty,'l2'
,dual,False
,tol,0.0001
,C,0.1
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,5000
,multi_class,'deprecated'


In [ ]:
#model performance using the hyperparameters given by GridSearch for F1-score
best_lr_f1 = gridsearch.best_estimator_

y_pred_f1 = best_lr_f1.predict(X_test)
y_prob_f1 = best_lr_f1.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_f1))
print(f'\nTuned ROC-AUC: {roc_auc_score(y_test, y_prob_f1)}')

              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1035
           1       0.50      0.79      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


Tuned ROC-AUC: 0.8375287400862848


In [ ]:
#use GridSearch to find best hyperparameters to maximize Recall-score
param_grid = {'C': [0.01, 0.1, 1, 10,],
              'penalty': ['l1', 'l2'],
              'solver': ['liblinear'],
              'class_weight': [None, 'balanced']}

log_reg_grid = LogisticRegression(max_iter= 5000, random_state= 42)

gridsearch = GridSearchCV(log_reg_grid, param_grid, scoring= 'recall', cv= 5, n_jobs= -1, verbose= 2)

gridsearch.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


,estimator,LogisticRegre...ndom_state=42)
,param_grid,"{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'penalty': ['l1', 'l2'], 'solver': ['liblinear']}"
,scoring,'recall'
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,penalty,'l1'


In [42]:
gridsearch.best_params_

{'C': 1, 'class_weight': 'balanced', 'penalty': 'l1', 'solver': 'liblinear'}

In [43]:
gridsearch.best_score_

np.float64(0.7979933110367894)

In [44]:
gridsearch.best_estimator_

,penalty,'l1'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,42
,solver,'liblinear'
,max_iter,5000
,multi_class,'deprecated'


In [ ]:
#model performance using the hyperparameters given by GridSearch for Recall-score
best_lr_recall = gridsearch.best_estimator_

y_pred_recall = best_lr_recall.predict(X_test)
y_prob_recall = best_lr_recall.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_recall))
print(f'\nTuned ROC-AUC: {roc_auc_score(y_test, y_prob_recall)}')

              precision    recall  f1-score   support

           0       0.90      0.71      0.80      1035
           1       0.50      0.79      0.61       374

    accuracy                           0.73      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.80      0.73      0.75      1409


Tuned ROC-AUC: 0.8365987238110001


In [ ]:
#table comparing the three models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def model_comparison(name: str, y_true, model_predictions, y_prob):
    return{'Name': name, 
           'Accuracy': accuracy_score(y_true, model_predictions),
           'Precision': precision_score(y_true, model_predictions),
           'Recall': recall_score(y_true, model_predictions),
           'F1 Score': f1_score(y_true, model_predictions),
           'ROC-AUC': roc_auc_score(y_test, y_prob)}

results = []
results.append(model_comparison('Original Logistic Model', y_test, predictions, probabilities))
results.append(model_comparison('F1 GridSearch', y_test, y_pred_f1, y_prob_f1))
results.append(model_comparison('Recall GridSearch', y_test, y_pred_recall, y_prob_recall))

results_df  = pd.DataFrame(results)
results_df

,Name,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Original Logistic Model,0.733144,0.498299,0.783422,0.609148,0.836229
1,F1 GridSearch,0.737402,0.503425,0.786096,0.613779,0.837529
2,Recall GridSearch,0.733854,0.499151,0.786096,0.610592,0.836599


### Key Observations

Hyperparameter tuning resulted in only small improvements over the original logistic regression model. The F1-optimized GridSearch shows slightly better performance across most metrics, but the gains are not too significant.

### Choosing the Right Model

For churn prediction, **recall is our priority metric** as we want to identify as many customers at risk of churning as possible, even if it means some false positives. Missing a customer who will actually churn is more costly than incorrectly flagging one who won't.

Therefore, I will proceed with the **Recall GridSearch model** as our base and focus on **threshold tuning** to further improve recall while balancing precision.

In [54]:
#Threshold tuning results
import numpy as np

#we just want result for churners (value = 1)
y_probabilities = best_lr_recall.predict_proba(X_test)[:, 1]

threshold = np.arange(0.1, 0.9, 0.05)
result = []

for prob in threshold:
    y_pred_prob = (y_probabilities >= prob).astype(int)

    result.append({'threshold': prob,
                   'precision': precision_score(y_test, y_pred_prob),
                   'recall': recall_score(y_test, y_pred_prob),
                   'f1': f1_score(y_test, y_pred_prob)})
    
threshold_df = pd.DataFrame(result)
threshold_df

,threshold,precision,recall,f1
0,0.10,0.339416,0.994652,0.506122
1,0.15,0.367944,0.975936,0.534407
2,0.20,0.389071,0.951872,0.552366
3,0.25,0.411489,0.938503,0.572127
4,0.30,0.429274,0.933155,0.588037
5,0.35,0.451187,0.914439,0.604240
6,0.40,0.468795,0.863636,0.607714
7,0.45,0.491987,0.820856,0.615230
8,0.50,0.499151,0.786096,0.610592
9,0.55,0.524621,0.740642,0.614191


In [49]:
# Best F1 threshold
best_f1_row = threshold_df.loc[threshold_df['f1'].idxmax()]
best_f1_row

threshold    0.600000
precision    0.557447
recall       0.700535
f1           0.620853
Name: 10, dtype: float64

## Selecting the Optimal Threshold

After analyzing different probability thresholds, I selected **0.35** as the optimal cutoff for our churn prediction model. Here's the reasoning behind this decision.

Lowering the threshold catches more churners but increases false alarms. Raising the threshold reduces false alarms but misses more churners. For churn prediction, missing a customer who will actually churn is more costly than offering a retention discount to someone who wouldn't have left.

Thresholds below 0.35 catch 93-99% of churners but at a significant cost. Precision drops to 33-43%, meaning more than half of our retention offers would go to customers who weren't actually at risk.

At 0.40, recall drops to 86% - we immediately miss 5% more churners. At 0.60 (the F1-optimized threshold), recall falls to 70%, meaning we miss nearly one in three customers who will actually churn. For a retention-focused business problem, this is too risky.

### Why 0.35?

At threshold 0.35, we achieve 91% recall while maintaining 45% precision. This means we catch nine out of ten customers who will actually churn, while about half of our retention efforts are accurately targeted. This balance maximizes customer retention while keeping false positives manageable.

In [ ]:
#result of the model using optimal threshold
BEST_THRESHOLD = 0.35

y_pred_final = (y_probabilities >= BEST_THRESHOLD).astype(int)

print(classification_report(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_probabilities))

              precision    recall  f1-score   support

           0       0.95      0.60      0.73      1035
           1       0.45      0.91      0.60       374

    accuracy                           0.68      1409
   macro avg       0.70      0.76      0.67      1409
weighted avg       0.82      0.68      0.70      1409

ROC-AUC: 0.8365987238110001
